Carregar os dados preparados:

In [1]:
# Importa a biblioteca Joblib para carregar os dados preparados.
import joblib

# Carrega os embeddings e as características estruturadas preparados no notebook anterior.
dados_preparados_embb = joblib.load(
    "../data/processed/dados_preparados_embb.joblib"
)

# Recupera os dados de treinamento contendo embeddings e características estruturadas.
X_treino_embb = dados_preparados_embb["X_treino_embb"]

# Recupera os dados de teste contendo embeddings e características estruturadas.
X_teste_embb = dados_preparados_embb["X_teste_embb"]

# Recupera os valores reais da variável-alvo do conjunto de treinamento.
y_treino = dados_preparados_embb["y_treino"]

# Recupera os valores reais da variável-alvo do conjunto de teste.
y_teste = dados_preparados_embb["y_teste"]

# Exibe o formato dos dados de treinamento para confirmar a quantidade de amostras e características.
print("Treinamento:", X_treino_embb.shape)

# Exibe o formato dos dados de teste para confirmar a quantidade de amostras e características.
print("Teste:", X_teste_embb.shape)

Treinamento: (1077, 408)
Teste: (270, 408)


Importar os modelos

| Modelo              | Vamos usar? | Motivo                                     |
| ------------------- | ----------- | ------------------------------------------ |
| Logistic Regression | ✅           | Adequado para embeddings                   |
| LinearSVC           | ✅           | Adequado para embeddings                   |
| Random Forest       | ✅           | Adequado para embeddings                   |
| XGBoost             | ✅           | Adequado para embeddings                   |
| MultinomialNB       | ⚠️          | Embeddings podem possuir valores negativos |


In [2]:
# Importa o classificador Logistic Regression.
from sklearn.linear_model import LogisticRegression

# Importa o classificador LinearSVC.
from sklearn.svm import LinearSVC

# Importa o classificador Random Forest.
from sklearn.ensemble import RandomForestClassifier

# Importa o classificador XGBoost.
from xgboost import XGBClassifier

# Importa o método utilizado para calibrar o LinearSVC e disponibilizar probabilidades.
from sklearn.calibration import CalibratedClassifierCV

Criar os modelos

In [3]:
# Cria o modelo de Regressão Logística para classificação das mensagens.
modelo_logistic_regression_embb = LogisticRegression(
    # Permite que o algoritmo realize mais iterações para encontrar uma solução.
    max_iter=1000,
    # Define uma semente para tornar o treinamento reproduzível.
    random_state=42
)

# Cria o LinearSVC que será utilizado como classificador base.
modelo_linear_svc_base_embb = LinearSVC(
    # Define uma semente para tornar o treinamento reproduzível.
    random_state=42
)

# Cria o LinearSVC calibrado para permitir a obtenção de probabilidades posteriormente.
modelo_linear_svc_embb = CalibratedClassifierCV(
    # Informa o LinearSVC que será calibrado.
    estimator=modelo_linear_svc_base_embb,
    # Utiliza cinco partes dos dados para realizar a calibração.
    cv=5
)

# Cria o modelo Random Forest para classificação das mensagens.
modelo_random_forest_embb = RandomForestClassifier(
    # Define a quantidade de árvores utilizadas pelo modelo.
    n_estimators=200,
    # Define uma semente para tornar o treinamento reproduzível.
    random_state=42,
    # Permite utilizar todos os processadores disponíveis.
    n_jobs=-1
)

# Cria o modelo XGBoost para classificação das mensagens.
modelo_xgboost_embb = XGBClassifier(
    # Define a quantidade de árvores utilizadas pelo modelo.
    n_estimators=200,
    # Define a profundidade máxima das árvores.
    max_depth=6,
    # Define a taxa de aprendizado utilizada pelo modelo.
    learning_rate=0.1,
    # Define a métrica utilizada durante o treinamento.
    eval_metric="logloss",
    # Define uma semente para tornar o treinamento reproduzível.
    random_state=42,
    # Permite utilizar todos os processadores disponíveis.
    n_jobs=-1
)

Treinar os modelos

In [4]:
# Treina a Regressão Logística utilizando os embeddings e as características estruturadas.
modelo_logistic_regression_embb.fit(
    X_treino_embb,
    y_treino
)

# Treina o LinearSVC calibrado utilizando os embeddings e as características estruturadas.
modelo_linear_svc_embb.fit(
    X_treino_embb,
    y_treino
)

# Treina o Random Forest utilizando os embeddings e as características estruturadas.
modelo_random_forest_embb.fit(
    X_treino_embb,
    y_treino
)

# Treina o XGBoost utilizando os embeddings e as características estruturadas.
modelo_xgboost_embb.fit(
    X_treino_embb,
    y_treino
)

# Exibe uma mensagem indicando que o treinamento dos quatro modelos foi concluído.
print("Treinamento dos modelos concluído com sucesso!")

Treinamento dos modelos concluído com sucesso!


Gerar as previsões

In [5]:
# Gera as classificações previstas pela Regressão Logística para as mensagens de teste.
previsoes_logistic_regression_embb = modelo_logistic_regression_embb.predict(
    X_teste_embb
)

# Gera as classificações previstas pelo LinearSVC calibrado para as mensagens de teste.
previsoes_linear_svc_embb = modelo_linear_svc_embb.predict(
    X_teste_embb
)

# Gera as classificações previstas pelo Random Forest para as mensagens de teste.
previsoes_random_forest_embb = modelo_random_forest_embb.predict(
    X_teste_embb
)

# Gera as classificações previstas pelo XGBoost para as mensagens de teste.
previsoes_xgboost_embb = modelo_xgboost_embb.predict(
    X_teste_embb
)

# Exibe a quantidade de previsões produzidas pela Regressão Logística.
print("Logistic Regression:", len(previsoes_logistic_regression_embb))

# Exibe a quantidade de previsões produzidas pelo LinearSVC.
print("LinearSVC:", len(previsoes_linear_svc_embb))

# Exibe a quantidade de previsões produzidas pelo Random Forest.
print("Random Forest:", len(previsoes_random_forest_embb))

# Exibe a quantidade de previsões produzidas pelo XGBoost.
print("XGBoost:", len(previsoes_xgboost_embb))

Logistic Regression: 270
LinearSVC: 270
Random Forest: 270
XGBoost: 270


                    EMBEDDINGS
                        ↓
                 384 dimensões
                        +
              24 features estruturadas
                        ↓
                 408 features
                        ↓
          ┌─────────────┼─────────────┬─────────────┐
          ↓             ↓             ↓             ↓
      Logistic       LinearSVC     Random Forest  XGBoost
      Regression     calibrado
          ↓             ↓             ↓             ↓
      Previsões      Previsões     Previsões     Previsões
          └─────────────┼─────────────┴─────────────┘
                        ↓
                   AVALIAÇÃO
                        ↓
          Accuracy / Precision / Recall
                 F1-score / ROC-AUC

Calcular Accuracy, Precision, Recall e F1-score

In [7]:
# Importa a biblioteca Pandas para criação e manipulação de DataFrames.
import pandas as pd

In [8]:
# Importa as métricas utilizadas para avaliar os modelos de classificação.
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Cria uma lista vazia para armazenar as métricas de cada modelo.
resultados_metricas_embb = []

# Define os nomes dos quatro modelos que serão comparados.
nomes_modelos_embb = [
    "Logistic Regression",
    "LinearSVC",
    "Random Forest",
    "XGBoost"
]

# Define as previsões produzidas pelos quatro modelos.
previsoes_modelos_embb = [
    previsoes_logistic_regression_embb,
    previsoes_linear_svc_embb,
    previsoes_random_forest_embb,
    previsoes_xgboost_embb
]

# Percorre os quatro modelos juntamente com suas respectivas previsões.
for nome_modelo, previsoes in zip(
    nomes_modelos_embb,
    previsoes_modelos_embb
):

    # Calcula a proporção de classificações corretas realizadas pelo modelo.
    accuracy = accuracy_score(
        y_teste,
        previsoes
    )

    # Calcula a proporção de previsões positivas que realmente eram possíveis golpes.
    precision = precision_score(
        y_teste,
        previsoes
    )

    # Calcula a proporção de possíveis golpes que foram identificados corretamente.
    recall = recall_score(
        y_teste,
        previsoes
    )

    # Calcula a média harmônica entre Precision e Recall.
    f1 = f1_score(
        y_teste,
        previsoes
    )

    # Adiciona as métricas calculadas à lista de resultados.
    resultados_metricas_embb.append([
        accuracy,
        precision,
        recall,
        f1
    ])

# Converte os resultados das métricas em um DataFrame.
df_metricas_embb = pd.DataFrame(
    resultados_metricas_embb,
    columns=[
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score"
    ],
    index=nomes_modelos_embb
)

# Exibe a tabela de comparação dos modelos utilizando embeddings.
df_metricas_embb

,Accuracy,Precision,Recall,F1-score
Logistic Regression,0.959259,0.969388,0.922330,0.945274
LinearSVC,0.962963,0.960396,0.941748,0.950980
Random Forest,0.944444,0.940000,0.912621,0.926108
XGBoost,0.970370,0.970297,0.951456,0.960784


06_preparacao_embb
        ↓
07_modelagem_embb
        ↓
Accuracy
Precision
Recall
F1
        ↓
ROC-AUC
        ↓
Thresholds
        ↓
Comparação final
        ↓
TF-IDF × Embeddings

Calcular ROC-AUC

In [9]:
# Importa a função roc_auc_score para calcular a área sob a curva ROC.
from sklearn.metrics import roc_auc_score

# Cria uma lista vazia para armazenar os resultados de ROC-AUC dos modelos.
resultados_auc_embb = []

# Define as probabilidades da classe "Possível golpe" produzidas pelos quatro modelos.
probabilidades_modelos_embb = [
    modelo_logistic_regression_embb.predict_proba(X_teste_embb)[:, 1],
    modelo_linear_svc_embb.predict_proba(X_teste_embb)[:, 1],
    modelo_random_forest_embb.predict_proba(X_teste_embb)[:, 1],
    modelo_xgboost_embb.predict_proba(X_teste_embb)[:, 1]
]

# Percorre cada modelo juntamente com suas respectivas probabilidades.
for nome_modelo, probabilidades in zip(
    nomes_modelos_embb,
    probabilidades_modelos_embb
):

    # Calcula o ROC-AUC comparando as classes reais com as probabilidades previstas.
    auc = roc_auc_score(
        y_teste,
        probabilidades
    )

    # Armazena o nome do modelo e seu respectivo ROC-AUC.
    resultados_auc_embb.append([
        nome_modelo,
        auc
    ])

# Cria um DataFrame contendo o ROC-AUC dos quatro modelos.
df_auc_embb = pd.DataFrame(
    resultados_auc_embb,
    columns=[
        "Modelo",
        "ROC-AUC"
    ]
)

# Exibe os valores de ROC-AUC dos modelos utilizando embeddings.
df_auc_embb

,Modelo,ROC-AUC
0,Logistic Regression,0.992966
1,LinearSVC,0.988547
2,Random Forest,0.989623
3,XGBoost,0.996919


Agora vamos testar os thresholds

Salvar os modelos com embeddings

In [10]:
# Importa a biblioteca Joblib para salvar os modelos treinados em arquivos.
import joblib

# Salva o modelo Logistic Regression treinado com embeddings.
joblib.dump(
    modelo_logistic_regression_embb,
    "../models/modelo_logistic_regression_embb.joblib"
)

# Salva o modelo LinearSVC calibrado treinado com embeddings.
joblib.dump(
    modelo_linear_svc_embb,
    "../models/modelo_linear_svc_embb.joblib"
)

# Salva o modelo Random Forest treinado com embeddings.
joblib.dump(
    modelo_random_forest_embb,
    "../models/modelo_random_forest_embb.joblib"
)

# Salva o modelo XGBoost treinado com embeddings.
joblib.dump(
    modelo_xgboost_embb,
    "../models/modelo_xgboost_embb.joblib"
)

# Exibe uma mensagem confirmando que os quatro modelos foram salvos.
print("Modelos com embeddings salvos com sucesso!")

Modelos com embeddings salvos com sucesso!
